# Getting the Silver Data and Pushing to Gold

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
df = spark.read.table("ecom.silver.fact_silver_orderreviews")
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
table_name = "ecom.gold.fact_gold_orderreviews"
silver_df = df.select(
    "reviewid",
    "orderid",
    "reviewscore",
    "reviewcommenttitle",
    "reviewcommentmessage",
    "review_sentiment",
    "reviewcreationdate",
    "reviewanswertimestamp",
    "source_system",
    "file_date",
    "ingest_date",
    "created_timestamp",
    "updated_timestamp"
).dropDuplicates(["reviewid"])
 

In [0]:

if spark.catalog.tableExists(table_name):

    gold_delta = DeltaTable.forName(spark, table_name)

    (
        gold_delta.alias("target")
        .merge(
            silver_df.alias("source"),
            "target.reviewid = source.reviewid"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )